### Phase B: The LLM Reasoning Engine

**Methodology:**
A pure RL policy acts as a 'black box'. For a public transit authority to trust an AI dispatch system, its decisions must be interpretable. This notebook implements an LLM layer (via Groq/Llama3) that translates the raw economic state and the PPO agent's action integer into a human-readable justification. This ensures transparent, accountable AI governance.

In [ ]:
!pip install groq

In [ ]:
import os

try:
    import groq
except ImportError:
    groq = None

def explain_action(bus_id, action, econ_data):
    action_str = {0: "PROCEED", 1: "HOLD", 2: "SKIP"}.get(action, "UNKNOWN")
    prompt = f"Bus {bus_id} chose to {action_str}. Economic calculation: Wait cost=₹{econ_data['wait_cost']:.0f}, Fuel cost=₹{econ_data['fuel_cost']:.0f}, Revenue=₹{econ_data['revenue']:.0f}. Explain in 1 sentence why this was the economically optimal decision."
    
    api_key = os.environ.get("GROQ_API_KEY")
    if api_key is None or groq is None:
        return f"Fallback: Bus {bus_id} chose {action_str} to maximize network economic efficiency and balance revenue against wait/fuel penalties."
        
    try:
        client = groq.Groq(api_key=api_key)
        resp = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=80
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return f"Fallback: Bus {bus_id} chose {action_str} to maximize network economic efficiency (API Error: {e})."

dummy_econ_data = {
    'wait_cost': 45.0,
    'fuel_cost': 22.0,
    'revenue': 150.0
}
explanation = explain_action(bus_id=2, action=0, econ_data=dummy_econ_data)
print("--- LLM Reasoning Output ---")
print(explanation)